# Clean up the web-search blocklist lab

Remove the dedicated lab API and, if this lab created it, the backend. Keep the shared APIM service, resource group, reused backend, Foundry account, and model deployments. APIM identities and Foundry role assignments are also retained because other APIs may share them.

Use the same environment file and API name as the main notebook. This notebook reads the deployment record to identify whether a backend was created, so changing `BACKEND_ID` after deployment does not change the cleanup target.

In [ ]:
from src.lab import load_config, service_parts, az_json

# Use the same selection as in the main notebook.
env_file = None
config = load_config(env_file)
service_id, subscription, resource_group, service_name = service_parts(config)
deployment_name = config.get("APIM_API_NAME", "web-search-blocklist")
deployment = az_json(
    "deployment", "group", "show", "--subscription", subscription,
    "--resource-group", resource_group, "--name", deployment_name,
)
parameters = deployment["properties"]["parameters"]
api_name = parameters["apiName"]["value"]
if parameters["apimServiceName"]["value"] != service_name:
    raise ValueError("The selected deployment belongs to a different APIM service.")
backend_created = not parameters["backendId"]["value"]
print("API to delete:", api_name)
print("Lab backend to delete:", f"{api_name}-foundry" if backend_created else "None; existing backend was reused")

## Delete the listed lab resources

Run the next cell after checking the API and backend names above. It does not delete the resource group or shared service. The deployment record is retained for auditing.

In [ ]:
management_url = f"https://management.azure.com{service_id}"
az_json(
    "rest", "--method", "delete", "--url",
    f"{management_url}/apis/{api_name}?api-version=2024-05-01&deleteRevisions=true",
    "--headers", "If-Match=*",
)
if backend_created:
    az_json(
        "rest", "--method", "delete", "--url",
        f"{management_url}/backends/{api_name}-foundry?api-version=2024-05-01",
        "--headers", "If-Match=*",
    )
print("Deleted the lab API and any lab-created backend. Shared resources retained.")